In [3]:
import scipy.io
import os

mat_dir = '../Exercise-Recognition-from-Wearable-Sensors/'
mat_file = os.path.join(mat_dir, 'exercise_data.50.0000_multionly.mat')

mat_data = scipy.io.loadmat(mat_file)

print("顶层变量 keys:", mat_data.keys())

顶层变量 keys: dict_keys(['__header__', '__version__', '__globals__', 'subject_data', 'exerciseConstants', 'Fs'])


In [17]:
import numpy as np

# 获取第一个受试者的所有记录
subj1 = mat_data['subject_data'][0, 0]   # shape (N,)

# 只看第一条 record
rec0 = subj1[0]
print("record[0] 的字段:", rec0.dtype.names)

act_mat = rec0['activityStartMatrix']
data_arr = rec0['data']

print("\nactivityStartMatrix:")
print("  type:", type(act_mat))
print("  shape:", act_mat.shape)
print("  dtype:", act_mat.dtype)

# 如果是一维对象数组，检查第一个元素
if act_mat.dtype == np.object_ and len(act_mat) > 0:
    elem = act_mat[0]
    print("  第一个元素 type:", type(elem), "shape:", elem.shape)
    # 如果是二维，查看第一行
    if isinstance(elem, np.ndarray) and elem.ndim == 2:
        print("  第一行内容:", elem[0])

print("\ndata:")
print("  type:", type(data_arr))
print("  shape:", data_arr.shape)
print("  dtype:", data_arr.dtype)

if data_arr.dtype == np.object_ and len(data_arr) > 0:
    elem = data_arr[0]
    print("  第一个元素 type:", type(elem))
    if hasattr(elem, 'dtype'):
        print("  第一个元素 dtype names:", elem.dtype.names)
        # 查看 accelDataMatrix 的形状
        if 'accelDataMatrix' in elem.dtype.names:
            acc = elem['accelDataMatrix']
            print("  accelDataMatrix shape:", acc.shape)

record[0] 的字段: ('masterFileToken', 'fileIndex', 'data', 'subjectIndex', 'activityIndex', 'instanceIndex', 'activityName', 'boundingWindow', 'instanceIndexInSourceFile', 'sampleRate', 'incompleteData', 'activityStartMatrix', 'subjectID', 'recordingID', 'masterToken')

activityStartMatrix:
  type: <class 'numpy.ndarray'>
  shape: (5,)
  dtype: object
  第一个元素 type: <class 'numpy.ndarray'> shape: (33, 7)
  第一行内容: [array(['<Initial Activity>'], dtype='<U18') array([[-15.207]])
 array([[103.019]]) array([], dtype='<U1') array([[-1]], dtype=int16)
 array([[0.012]])
 array([[(array([[nan]]), array([[28353]], dtype=uint16), array([[-1]], dtype=int16), array([[-1]], dtype=int16), array([[nan]]))]],
       dtype=[('startSequenceNumberMaster', 'O'), ('endSequenceNumberMaster', 'O'), ('startSequenceNumberSlave', 'O'), ('endSequenceNumberSlave', 'O'), ('slaveImuOffset_local', 'O')])]

data:
  type: <class 'numpy.ndarray'>
  shape: (5,)
  dtype: object
  第一个元素 type: <class 'numpy.ndarray'>
  第一个元素 dt

In [19]:
import numpy as np
import pandas as pd
import os

subject_data = mat_data['subject_data']
num_subjects = subject_data.shape[0]

all_records = []
label_map = {}
current_label = 0

skip_activities = {'<Initial Activity>', 'Tap Left Device', 'Tap Right Device',
                   'Non-Exercise', 'Device on Table', 'Arm Band Adjustment'}

for subj_idx in range(num_subjects):
    records = subject_data[subj_idx, 0]   # shape (N,)
    for rec_idx in range(records.shape[0]):
        record = records[rec_idx]
        
        # 获取活动名称矩阵和传感器数据
        act_mat_1d = record['activityStartMatrix']  # shape (K,) object
        data_1d = record['data']                    # shape (K,) object
        
        # 提取受试者ID
        subj_id = record['subjectID']
        while isinstance(subj_id, np.ndarray) and subj_id.size > 0:
            subj_id = subj_id.ravel()[0]
        try:
            subj_id = int(subj_id)
        except:
            pass
        
        # 片段数量
        num_segments = min(len(act_mat_1d), len(data_1d))
        
        # 遍历每个片段
        for seg_idx in range(num_segments):
            # 活动块 (M, 7)
            act_block = act_mat_1d[seg_idx]
            # 传感器数据
            data_struct = data_1d[seg_idx]
            # 真正的加速度和陀螺仪矩阵被包装在 (1,1) 的数组里
            acc_full = data_struct['accelDataMatrix'][0, 0]
            gyr_full = data_struct['gyroDataMatrix'][0, 0]
            
            if acc_full.ndim != 2 or acc_full.shape[1] < 4:
                continue
            if gyr_full.ndim != 2 or gyr_full.shape[1] < 4:
                continue
            
            # 遍历 act_block 中的每一行（每个子活动）
            for row_idx in range(act_block.shape[0]):
                # 活动名称
                name_arr = act_block[row_idx, 0]
                while isinstance(name_arr, np.ndarray) and name_arr.size > 0:
                    name_arr = name_arr.ravel()[0]
                act_name = str(name_arr)
                
                if act_name in skip_activities:
                    continue
                
                # 起止帧号（在最后一列的结构体中）
                seq_info = act_block[row_idx, -1]
                start_seq = seq_info['startSequenceNumberMaster']
                end_seq   = seq_info['endSequenceNumberMaster']
                # 安全提取标量值
                while isinstance(start_seq, np.ndarray) and start_seq.size > 0:
                    start_seq = start_seq.ravel()[0]
                while isinstance(end_seq, np.ndarray) and end_seq.size > 0:
                    end_seq = end_seq.ravel()[0]
                
                start_idx = int(start_seq) - 1
                end_idx   = int(end_seq) - 1
                
                if start_idx < 0 or end_idx >= acc_full.shape[0] or end_idx <= start_idx:
                    continue
                
                acc_seg = acc_full[start_idx:end_idx+1, 1:]  # 去掉时间戳
                gyr_seg = gyr_full[start_idx:end_idx+1, 1:]
                
                # 标签映射
                if act_name not in label_map:
                    label_map[act_name] = current_label
                    current_label += 1
                lbl = label_map[act_name]
                
                df_temp = pd.DataFrame({
                    'acc_x': acc_seg[:, 0],
                    'acc_y': acc_seg[:, 1],
                    'acc_z': acc_seg[:, 2],
                    'gyro_x': gyr_seg[:, 0],
                    'gyro_y': gyr_seg[:, 1],
                    'gyro_z': gyr_seg[:, 2],
                    'label': lbl,
                    'subject_id': subj_id
                })
                all_records.append(df_temp)

if all_records:
    df_final = pd.concat(all_records, ignore_index=True)
    print("✅ 提取完成！")
    print(f"总行数：{len(df_final)}")
    print(f"\n健身动作映射：")
    for name, lid in sorted(label_map.items(), key=lambda x: x[1]):
        print(f"  {lid} → {name}")
    print(f"\n各标签数据量：")
    print(df_final['label'].value_counts().sort_index())
    
    os.makedirs('../data/processed/', exist_ok=True)
    df_final.to_csv('../data/processed/recofit_processed.csv', index=False)
    print("\n✅ 已保存到 data/processed/recofit_processed.csv")
else:
    print("❌ 没有提取到任何数据")

✅ 提取完成！
总行数：5392737

健身动作映射：
  0 → Jumping Jacks
  1 → Plank
  2 → Squat Jump
  3 → Sit-ups
  4 → Dumbbell Deadlift Row
  5 → Side Plank Right side
  6 → Side Plank Left side
  7 → Bicep Curl
  8 → Pushups
  9 → Walking lunge
  10 → Power Boat pose
  11 → Squat
  12 → Pushup (knee or foot variation)
  13 → Squat (arms in front of body, parallel to ground)
  14 → Overhead Triceps Extension
  15 → Sit-up (hands positioned behind head)
  16 → Dip
  17 → Unlisted Exercise
  18 → Repetitive Stretching
  19 → Shoulder Press (dumbbell)
  20 → V-up
  21 → Dumbbell Row (knee on bench) (label spans both arms)
  22 → Lunge (alternating both legs, weight optional)
  23 → Two-arm Dumbbell Curl (both arms, not alternating)
  24 → Burpee
  25 → Triceps Kickback (knee on bench) (label spans both arms)
  26 → Static stretch
  27 → Kettlebell Swing
  28 → Russian Twist
  29 → Wall Squat
  30 → Lateral Raise
  31 → Static Stretch (at your own pace)
  32 → Squat Rack Shoulder Press
  33 → Crunch
  34 → Tr

In [22]:
import pandas as pd

df = pd.read_csv('../data/processed/recofit_processed.csv')

# 已知映射（从之前的输出复制）
label_names = {
    0: 'Jumping Jacks', 1: 'Plank', 2: 'Squat Jump', 3: 'Sit-ups', 4: 'Dumbbell Deadlift Row',
    5: 'Side Plank Right side', 6: 'Side Plank Left side', 7: 'Bicep Curl', 8: 'Pushups',
    9: 'Walking lunge', 10: 'Power Boat pose', 11: 'Squat', 12: 'Pushup (knee or foot variation)',
    13: 'Squat (arms in front of body, parallel to ground)', 14: 'Overhead Triceps Extension',
    15: 'Sit-up (hands positioned behind head)', 16: 'Dip', 17: 'Unlisted Exercise',
    18: 'Repetitive Stretching', 19: 'Shoulder Press (dumbbell)', 20: 'V-up',
    21: 'Dumbbell Row (knee on bench) (label spans both arms)',
    22: 'Lunge (alternating both legs, weight optional)',
    23: 'Two-arm Dumbbell Curl (both arms, not alternating)', 24: 'Burpee',
    25: 'Triceps Kickback (knee on bench) (label spans both arms)', 26: 'Static stretch',
    27: 'Kettlebell Swing', 28: 'Russian Twist', 29: 'Wall Squat', 30: 'Lateral Raise',
    31: 'Static Stretch (at your own pace)', 32: 'Squat Rack Shoulder Press', 33: 'Crunch',
    34: 'Triceps extension (lying down)', 35: 'Biceps Curl (band)', 36: 'Elliptical machine',
    37: 'Band Pull-Down Row', 38: 'Chest Press (rack)', 39: 'Squat (kettlebell / goblet)',
    40: 'Rowing machine', 41: 'Wall Ball', 42: 'Running (treadmill)', 43: 'Medicine Ball Slam',
    44: 'Lawnmower (right arm)', 45: 'Lawnmower (left arm)', 46: 'Squat (hands behind head)',
    47: 'Tap IMU Device', 48: 'Arm straight up', 49: 'Walk',
    50: 'Seated Back Fly', 51: 'Butterfly Sit-up', 52: 'Jump Rope',
    53: 'Fast Alternating Punches', 54: 'Dumbbell Squat (hands at side)',
    55: 'Dynamic Stretch (at your own pace)', 56: 'Box Jump (on bench)',
    57: 'Lawnmower (label spans both arms)', 58: 'Note',
    59: 'Dumbbell Row (knee on bench) (right arm)', 60: 'Dumbbell Row (knee on bench) (left arm)',
    61: 'Triceps Kickback (knee on bench) (right arm)', 62: 'Triceps Kickback (knee on bench) (left arm)',
    63: 'Overhead Triceps Extension (label spans both arms)', 64: 'Rest',
    65: 'Invalid', 66: 'Alternating Dumbbell Curl'
}

# 要移除的数字标签
remove_ids = [47, 48, 58, 64, 65]
removed_names = [label_names[i] for i in remove_ids]

print("移除了以下非运动/无效标签：")
for i, name in zip(remove_ids, removed_names):
    print(f"  {i}: {name}")

# 过滤
df_filtered = df[~df['label'].isin(remove_ids)].copy()

# 重新映射为连续整数 (0 ~ 61)
df_filtered['label'] = df_filtered['label'].astype('category').cat.codes

# 构建新的标签映射
# 获取新旧标签的对应关系
old_categories = df_filtered['label'].astype('category').cat.categories
new_mapping = {}
for new_id, old_id in enumerate(old_categories):
    new_mapping[new_id] = label_names[old_id]

print(f"\n移除后保留的动作种类: {len(new_mapping)}")
print("\n========== 完整标签映射 (新标签 -> 名称) ==========")
for new_id in sorted(new_mapping.keys()):
    print(f"  {new_id:2d} -> {new_mapping[new_id]}")

print(f"\n总行数（移除前后）: {len(df)} -> {len(df_filtered)}")

# 保存
import os
os.makedirs('../data/processed/', exist_ok=True)
df_filtered.to_csv('../data/processed/recofit_core.csv', index=False)
print("\n✅ 已保存到 data/processed/recofit_core.csv")

移除了以下非运动/无效标签：
  47: Tap IMU Device
  48: Arm straight up
  58: Note
  64: Rest
  65: Invalid

移除后保留的动作种类: 62

========== 完整标签映射 (新标签 -> 名称) ==========
   0 -> Jumping Jacks
   1 -> Plank
   2 -> Squat Jump
   3 -> Sit-ups
   4 -> Dumbbell Deadlift Row
   5 -> Side Plank Right side
   6 -> Side Plank Left side
   7 -> Bicep Curl
   8 -> Pushups
   9 -> Walking lunge
  10 -> Power Boat pose
  11 -> Squat
  12 -> Pushup (knee or foot variation)
  13 -> Squat (arms in front of body, parallel to ground)
  14 -> Overhead Triceps Extension
  15 -> Sit-up (hands positioned behind head)
  16 -> Dip
  17 -> Unlisted Exercise
  18 -> Repetitive Stretching
  19 -> Shoulder Press (dumbbell)
  20 -> V-up
  21 -> Dumbbell Row (knee on bench) (label spans both arms)
  22 -> Lunge (alternating both legs, weight optional)
  23 -> Two-arm Dumbbell Curl (both arms, not alternating)
  24 -> Burpee
  25 -> Triceps Kickback (knee on bench) (label spans both arms)
  26 -> Static stretch
  27 -> Kettlebell Sw